# Lab 05 — reproducible rubidium analysis

This notebook calls the portable Python implementation using the included measurements. See [the project README](../README.md), [provenance](../PROVENANCE.md) and [analysis review](../ANALYSIS_REVIEW.md). Fits describe four features in a difference trace; frequency widths, optical transmission, density and pressure remain unverified.

Install the project requirements and JupyterLab as described in [environment setup](../../../docs/ENVIRONMENT.md). Open from the repository, project or notebook directory. Running writes generated outputs to `runs/lab5-notebook/` in the repository.

In [ ]:
from pathlib import Path
import importlib.util
import json

candidates = [Path.cwd(), *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'projects/rubidium-absorption/src/analyze.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Run this notebook from inside the portfolio repository.')
PROJECT = REPO / 'projects/rubidium-absorption'
spec = importlib.util.spec_from_file_location('lab5_analysis', PROJECT / 'src/analyze.py')
lab5 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab5)
config = json.loads((PROJECT / 'config/analysis.json').read_text())
result = lab5.analyze(config, project_root=PROJECT)
OUTPUT = REPO / 'runs/lab5-notebook'
lab5.write_outputs(result, OUTPUT)
print(f'Outputs saved to {OUTPUT}')

In [ ]:
for fit in result['summary']['fits']:
    print(f"Feature {fit['feature']}: center {fit['center_ms']:.4f} ms; FWHM {fit['fwhm_ms']:.6f} ms")
print(json.dumps(result['summary']['transmission_review'], indent=2))

## Fit and residuals

Systematic residuals and strong window sensitivity limit the physical interpretation. The images below are loaded from this run's output folder.

In [ ]:
from PIL import Image
Image.open(OUTPUT / 'four-feature-fits.png')

In [ ]:
Image.open(OUTPUT / 'trace-overview.png')

In [ ]:
Image.open(OUTPUT / 'window-sensitivity.png')

## Review before interpreting

The selected background has dips resembling the cell trace, and the historical frequency mapping stretches one frequency span across multiple sweeps. The density calculation's raw voltage ratio is negative; the portable code retains that diagnostic without shifting or clamping it. Narrow-window widths are sensitivity checks, not corrected measurements.

The [archived notebook](lab5-archived.ipynb) retains the original code and results with attribution. Numerical tests can be run from the repository root with `python -m unittest discover -s projects/rubidium-absorption/tests -v`.